# 06. Time Series — кратко и практично

| Блок | Инструменты |
|------|-------------|
| Индекс и диапазоны | `date_range`, `to_datetime`, `DatetimeIndex` |
| Resampling | `.resample()` — OHLC, mean, ffill |
| Rolling / Expanding / EWM | скользящие метрики для ML-фичей |
| Shift & lag | `shift` — создание lag-фичей |
| Period & offset | `Period`, `MonthEnd` — когда нужны периоды, а не моменты |

## 1. DatetimeIndex — основа временного ряда

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# date_range: freq='B' = рабочие дни (без выходных)
dates = pd.date_range('2024-01-01', periods=10, freq='B')
prices = pd.Series(100 + np.cumsum(np.random.randn(10) * 0.8), index=dates, name='AAPL')
print(prices)
print(f'\ndtype индекса: {prices.index.dtype}')

# Удобная индексация по дате
print('\nОдин день:')
print(prices['2024-01-03'])
print('\nСрез по диапазону:')
print(prices['2024-01-08':'2024-01-12'])

# to_datetime — парсинг из строк/колонки df
raw_dates = ['2024-01-01', '2024-02-15', '2024-03-31']
parsed = pd.to_datetime(raw_dates)
print('\nto_datetime:', parsed)

## 2. Resampling — изменение частоты

`resample` = groupby по времени. Downsampling (день→неделя) или upsampling (день→час).

In [ ]:
np.random.seed(0)
dates = pd.date_range('2024-01-01', periods=60, freq='B')
close = pd.Series(100 + np.cumsum(np.random.randn(60) * 0.8), index=dates, name='close')
volume = pd.Series(np.random.randint(1_000_000, 10_000_000, 60), index=dates, name='volume')

# Downsampling: дневные → недельные OHLC (ключевой паттерн для финансов)
ohlc = close.resample('W').ohlc()
print('OHLC по неделям:')
print(ohlc.head())

# Среднее и суммарный объём по неделям
weekly = pd.DataFrame({
    'avg_close': close.resample('W').mean(),
    'total_volume': volume.resample('W').sum(),
})
print('\nНедельные агрегаты:')
print(weekly.head().round(2))

# Upsampling: добавить часовые точки и заполнить ffill
hourly = close.resample('h').ffill()
print(f'\nUpsampling день→час: {len(close)} → {len(hourly)} точек')

## 3. Rolling / Expanding / EWM — скользящие метрики

Главный источник ML-фичей из временного ряда.

| | Окно | Когда |
|---|---|---|
| `rolling(n)` | фиксированное n последних | волатильность, MA |
| `expanding()` | от начала до текущей точки | накопленная статистика |
| `ewm(span=n)` | экспоненциальное (свежее важнее) | EWMA цены, сигнала |

In [ ]:
daily_ret = close.pct_change().dropna()

# Rolling: скользящие фичи
features = pd.DataFrame(index=close.index)
features['close']      = close
features['ma_10']      = close.rolling(10).mean()            # скользящее среднее
features['ma_20']      = close.rolling(20).mean()
features['vol_10']     = daily_ret.rolling(10).std()         # скользящая волатильность
features['vol_20']     = daily_ret.rolling(20).std()
features['ret_5d']     = close.pct_change(5)                 # доходность за 5 дней

# Expanding: накопленная статистика
features['expanding_mean_vol'] = features['vol_10'].expanding(min_periods=10).mean()

# EWM: экспоненциальное (свежие данные важнее)
features['ewma_10']    = close.ewm(span=10).mean()

# min_periods — сколько точек нужно минимум чтобы считать (иначе NaN)
features['vol_min_p']  = daily_ret.rolling(20, min_periods=5).std()

print('ML-фичи из временного ряда:')
print(features.tail(8).round(4))

## 4. `shift` — lag-фичи и forward returns

`shift(n)` — сдвиг на n периодов. Положительный = лаг (прошлое), отрицательный = будущее.

**Lag-фичи** — классика ML на временных рядах: «что было вчера/неделю назад».
**Forward return** — таргет: «что будет через n дней» (shift с отрицательным n).

In [ ]:
df = pd.DataFrame({'close': close, 'ret': daily_ret})

# Lag-фичи (прошлое → признак)
df['ret_lag1'] = df['ret'].shift(1)   # вчерашняя доходность
df['ret_lag5'] = df['ret'].shift(5)   # 5 дней назад

# Forward return (будущее → таргет)
df['target_5d'] = df['close'].shift(-5) / df['close'] - 1  # доходность через 5 дней

print('Lag-фичи и таргет:')
print(df[['close','ret','ret_lag1','ret_lag5','target_5d']].iloc[4:12].round(4))
print('\nПоследние строки — target NaN (нет данных в будущем):')
print(df['target_5d'].tail())

## 5. Period и offsets — когда нужны периоды

`Timestamp` = момент времени. `Period` = промежуток (квартал, месяц).
Используется для финансовой отчётности, где данные привязаны к периоду, а не дате.

In [ ]:
from pandas.tseries.offsets import MonthEnd, BDay

# Period: квартальная прибыль
q1 = pd.Period('2024Q1', freq='Q')
print(f'Q1: {q1}, следующий: {q1 + 1}, начало: {q1.start_time.date()}, конец: {q1.end_time.date()}')

# period_range — для финансовой отчётности
quarters = pd.period_range('2023Q1', periods=6, freq='Q')
earnings = pd.Series([2.1, 2.4, 1.9, 2.8, 3.1, 2.7], index=quarters, name='EPS')
print('\nEPS по кварталам:')
print(earnings)

# MonthEnd offset — сдвинуть дату к концу месяца
mid_month = pd.Timestamp('2024-05-15')
print(f'\nКонец месяца от {mid_month.date()}: {MonthEnd().rollforward(mid_month).date()}')

# Добавить N рабочих дней (BDay)
settlement = mid_month + 2 * BDay()  # T+2 расчёты
print(f'T+2 от {mid_month.date()}: {settlement.date()}')

# to_timestamp / to_period — конвертация
print('\nPeriod → Timestamp (начало периода):')
print(earnings.to_timestamp().head(3))

## 6. Итоговый паттерн: временной ряд → ML feature matrix

In [ ]:
np.random.seed(99)
tickers = ['AAPL', 'GOOG', 'MSFT']
dates   = pd.date_range('2024-01-01', periods=120, freq='B')

# Wide: цены
prices = pd.DataFrame(
    100 + np.cumsum(np.random.randn(120, 3) * 0.8, axis=0),
    index=dates, columns=tickers
)

# Все фичи для каждого тикера
feature_parts = []
for t in tickers:
    p = prices[t]
    r = p.pct_change()
    feat = pd.DataFrame({
        f'{t}_ma10':      p.rolling(10).mean(),
        f'{t}_vol20':     r.rolling(20).std(),
        f'{t}_ewma10':    p.ewm(span=10).mean(),
        f'{t}_ret_lag1':  r.shift(1),
        f'{t}_ret_5d':    p.pct_change(5),
        f'{t}_target_5d': p.shift(-5) / p - 1,   # таргет: доходность через 5 дней
    })
    feature_parts.append(feat)

feature_matrix = pd.concat(feature_parts, axis=1).dropna()
print(f'Feature matrix: {feature_matrix.shape}')
print(feature_matrix.head(3).round(4))